# PGx Risk Calculator Dashboard – Full Deployment Workflow

**Purpose:** Deploy the PGx Risk Calculator Dashboard from cohorts with aggregated feature importances through Lambda/Docker.  
**Updated:** January 2026

## Overview

This notebook **consolidates the pipeline from Step 4 onward**. It **runs pipeline Step 4, 5, and 6 in this notebook** (see cells below), then prepares and deploys the dashboard:

- **Pipeline Step 4** — Model data: `4_model_data/create_model_data.py` (model_events.parquet per cohort/age_band)
- **Pipeline Step 5** — PGx analysis: `5_pgx_analysis/run_analysis.py` (PGx features added to model data)
- **Pipeline Step 6** — Final model training: `6_final_model/run_final_model.py` (trained models and feature_schema.json)

**Required inputs (must exist before running pipeline Step 4):**

- **Cohorts** (Step 2) — `gold/cohorts` cohort.parquet files (case/control and target dates)  
- **Feature importances** (Step 3/3b) — cohort_feature_importance CSVs and feature_filtering_summary.json, synced from S3 or under project/NVMe

**Required outputs (for deployment):**

- **SHAP** (Step 7) and **FFA** (Step 8) results — must be produced and combined for the Causal Analysis tab.

## Cohort / model mapping

| Model | PGx cohort | Age bands | Description |
|-------|------------|-----------|-------------|
| **Opioid ED** | `opioid_ed` | 13-24, 25-44, 45-54, 55-64 | Opioid-related ED visit predictive model |
| **Polypharmacy** | `non_opioid_ed` | 65-74, 75-84, 85-94 | Polypharmacy / adverse drug event model |

## Workflow Steps

1. **Sync inputs from S3 to NVMe** (idempotent) – Step 3a/3b feature importance (and optionally Step 6 if already built elsewhere).
2. **Verify inputs** – Feature importance (Step 3/3b) per cohort/age_band; Step 6 outputs if already present.
3. **Pipeline Step 4–6** – Run the **Pipeline Step 4**, **5**, and **6** cells below (model data → PGx analysis → final model training). Skip if Step 6 outputs already exist and are synced.
4. **Generate metadata** (idempotent, checkpoint) – Extract valid codes from feature importance for dashboard dropdowns.
5. **Prepare models** (idempotent, checkpoint) – Package models and feature schemas from Step 6 outputs.
6. **Combine SHAP/FFA** (required) – Run Step 7 (SHAP) and Step 8 (FFA), then combine results for Causal Analysis tab.
7. **Prepare Lambda directory** – Assemble `lambda_dir` for Docker build.
8. **Verify & deploy** – Verify `lambda_dir`, then build Docker image and deploy (ECR/API Gateway).

## Reference

- PGx data prep: `9_risk_dashboard/data_preparation/`

In [1]:
# Setup: paths and project root
import sys
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == "9_risk_dashboard":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / "9_risk_dashboard").exists():
    pass
else:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))
from py_helpers.env_utils import get_data_root
from py_helpers.workflow_sync_checkpoint import sync_s3_to_local, check_step_checkpoint_exists, save_step_checkpoint

DASHBOARD_DIR = PROJECT_ROOT / "9_risk_dashboard"
DATA_PREP_DIR = DASHBOARD_DIR / "data_preparation"
DEPLOY_DIR = DASHBOARD_DIR / "deployment"
S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")
DATA_ROOT = get_data_root()
AWS_PROFILE = os.environ.get("AWS_PROFILE")

print("PGx Risk Calculator Workflow")
print("=" * 60)
print(f"Project root: {PROJECT_ROOT}")
print(f"Dashboard dir: {DASHBOARD_DIR}")
print(f"Data prep: {DATA_PREP_DIR}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print("=" * 60)

PGx Risk Calculator Workflow
Project root: /home/pgx3874/pgx-analysis
Dashboard dir: /home/pgx3874/pgx-analysis/9_risk_dashboard
Data prep: /home/pgx3874/pgx-analysis/9_risk_dashboard/data_preparation
Data root (NVMe/local): /mnt/nvme


In [2]:
# Configuration: PGx cohorts and age bands (aligned with prepare_lambda_dir.py)
# opioid_ed (younger age bands); non_opioid_ed / polypharmacy (older age bands)
REQUIRED_COHORTS = {
    "opioid_ed": ["13-24", "25-44", "45-54", "55-64"],
    "non_opioid_ed": ["65-74", "75-84", "85-94"],
}

# Input dirs (required for pipeline Step 4–6)
# Cohorts: Step 2 cohort.parquet files (create_model_data reads case/control and target dates from here).
COHORTS_ROOT = DATA_ROOT / "gold" / "cohorts"
# Feature importance: Step 3/3b outputs — cohort_feature_importance.csv and feature_filtering_summary.json per cohort/age_band.
FI_ROOT = DATA_ROOT / "gold" / "feature_importance"
STEP3_OUTPUTS = STEP3B_OUTPUTS = FI_ROOT
# Model data: single canonical location (Step 4 output, Step 5/6 input).
from py_helpers.env_utils import get_model_data_root
MODEL_DATA_ROOT = get_model_data_root()

# Output dirs (Step 6 final model outputs; data prep and Lambda read from these)
FINAL_MODEL_OUTPUTS = PROJECT_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_OUTPUTS_ALT = DATA_ROOT / "6_final_model" / "outputs"
FINAL_MODEL_GOLD = DATA_ROOT / "gold" / "final_model"  # S3 layout: cohort/13-24/*.joblib

print("Cohorts and age bands:")
for cohort, bands in REQUIRED_COHORTS.items():
    print(f"  {cohort}: {bands}")
print("\nInput dirs (for Step 4–6):")
print(f"  Cohorts (2):              {COHORTS_ROOT}")
print(f"  Feature importance (3/3b): {FI_ROOT}  (CSVs + feature_filtering_summary.json)")
print(f"  Model data (4; in/out):   {MODEL_DATA_ROOT}")
print("\nOutput dirs (Step 6):")
print(f"  Project:   {FINAL_MODEL_OUTPUTS}")
print(f"  NVMe:      {FINAL_MODEL_OUTPUTS_ALT}")
print(f"  gold/NVMe: {FINAL_MODEL_GOLD}")

Cohorts and age bands:
  opioid_ed: ['13-24', '25-44', '45-54', '55-64']
  non_opioid_ed: ['65-74', '75-84', '85-94']

Input dirs (for Step 4–6):
  Cohorts (2):              /mnt/nvme/gold/cohorts
  Feature importance (3/3b): /mnt/nvme/gold/feature_importance  (CSVs + feature_filtering_summary.json)
  Model data (4; in/out):   /mnt/nvme/4_model_data

Output dirs (Step 6):
  Project:   /home/pgx3874/pgx-analysis/6_final_model/outputs
  NVMe:      /mnt/nvme/6_final_model/outputs
  gold/NVMe: /mnt/nvme/gold/final_model


## Sync required inputs from S3 to NVMe (idempotent)

Sync **cohorts** (Step 2), **feature importance** (Step 3/3b), and **Step 6** final model outputs from S3 so pipeline and data preparation can read from local/NVMe. **Idempotent:** `aws s3 sync` only updates changed or missing files.

In [3]:
# Sync cohorts (Step 2), Step 3a/3b feature importance, and Step 6 final models from S3 to NVMe (DATA_ROOT).
# Cohorts -> COHORTS_ROOT (gold/cohorts); Feature importance -> gold/feature_importance; Step 6 -> gold/final_model.
COHORTS_ROOT.mkdir(parents=True, exist_ok=True)
FI_SYNC_TARGET = DATA_ROOT / "gold" / "feature_importance"
FI_SYNC_TARGET.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_GOLD.mkdir(parents=True, exist_ok=True)

sync_s3_to_local(f"s3://{S3_BUCKET}/gold/cohorts/", COHORTS_ROOT, profile=AWS_PROFILE)
sync_s3_to_local(f"s3://{S3_BUCKET}/gold/feature_importance/", FI_SYNC_TARGET, profile=AWS_PROFILE)
sync_s3_to_local(f"s3://{S3_BUCKET}/gold/final_model/", FINAL_MODEL_GOLD, profile=AWS_PROFILE)
print("Sync complete. Run Step 0 verification below.")

Sync complete. Run Step 0 verification below.


## Step 0: Verify inputs (FI required; 4_model_data and Step 6 informational)

**Required:** **Feature importance** (Step 3/3b) — must exist for each cohort/age_band so Pipeline Step 4 can run.

**Informational:** **ModelData** checks `DATA_ROOT/4_model_data` and `PROJECT_ROOT/4_model_data` (same location `create_model_data.py` writes to). **Model** = Step 6 outputs. Both are produced by Pipeline Step 4–6 cells below; if already present, you can skip those cells.

In [4]:
def check_feature_importance(cohort: str, age_band: str) -> bool:
    ab = age_band.replace("-", "_")
    # Step 3b refined: FI_ROOT (NVMe) then project 3b/outputs
    for base in (STEP3B_OUTPUTS, PROJECT_ROOT / "3b_feature_importance_eda" / "outputs"):
        fi_3b = base / cohort / ab / f"{cohort}_{ab}_cohort_feature_importance.csv"
        if fi_3b.exists():
            return True
    # Step 3 aggregated: FI_ROOT then project 3a/outputs
    for base in (STEP3_OUTPUTS, PROJECT_ROOT / "3a_feature_importance" / "outputs"):
        fi_3 = base / cohort / ab / f"{cohort}_{ab}_aggregated_feature_importance.csv"
        if fi_3.exists():
            return True
    return False

def check_cohorts(cohort: str, age_band: str) -> bool:
    """Check Step 2 cohort.parquet exists for at least one year (2016–2019). Layout: COHORTS_ROOT/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet."""
    for year in (2016, 2017, 2018, 2019):
        p = COHORTS_ROOT / f"cohort_name={cohort}" / f"event_year={year}" / f"age_band={age_band}" / "cohort.parquet"
        if p.exists():
            return True
    return False

def check_model_data(cohort: str, age_band: str) -> bool:
    """Check model_events.parquet at canonical MODEL_DATA_ROOT (same location create_model_data.py writes to)."""
    p = MODEL_DATA_ROOT / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
    return p.exists()

def check_final_model(cohort: str, age_band: str) -> bool:
    ab = age_band.replace("-", "_")
    # 1) Project or DATA_ROOT/6_final_model/outputs: cohort/13_24/models/*.joblib
    for base in (FINAL_MODEL_OUTPUTS, FINAL_MODEL_OUTPUTS_ALT):
        model_dir = base / cohort / ab
        if not model_dir.exists():
            continue
        models_sub = model_dir / "models"
        if models_sub.exists() and any(models_sub.glob("*.joblib")):
            return True
        if (model_dir / "feature_schema.json").exists():
            return True
    # 2) DATA_ROOT/gold/final_model (S3-synced): cohort/13-24/*.joblib (hyphen in age_band)
    gold_dir = FINAL_MODEL_GOLD / cohort / age_band
    if gold_dir.exists() and any(gold_dir.glob("*.joblib")):
        return True
    return False

print("Step 0: Verify feature importance (required); cohorts and 4_model_data (Step 4 inputs); Step 6 (informational)\n")
print("  Locations: Cohorts=COHORTS_ROOT, FI=Step 3/3b, ModelData=MODEL_DATA_ROOT, Model=Step 6 outputs\n")
fi_ok_all = True
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        cohorts_ok = check_cohorts(cohort, age_band)
        fi_ok = check_feature_importance(cohort, age_band)
        model_data_ok = check_model_data(cohort, age_band)
        model_ok = check_final_model(cohort, age_band)
        if not fi_ok:
            fi_ok_all = False
        status = "ready" if fi_ok else "missing FI"
        print(f"  {cohort} / {age_band}:  Cohorts={cohorts_ok}, FI={fi_ok}, ModelData={model_data_ok}, Model={model_ok}  -> {status}")
if fi_ok_all:
    print("\nAll prerequisites are available to build model data. Run Pipeline Step 4–6 cells below.")
    print("  (If Step 6 is already built elsewhere, you can sync from S3 or skip those cells.)")
else:
    print("\nMissing feature importance for some cohort/age_band. Sync from S3 or run Step 3/3b first, then re-run this cell.")
if fi_ok_all:
    cohorts_missing = [(c, ab) for c, bands in REQUIRED_COHORTS.items() for ab in bands if not check_cohorts(c, ab)]
    if cohorts_missing:
        print("\nCohorts=False for some cohort/age_band. Sync gold/cohorts from S3 (run Sync cell) or run Step 2. Expected layout: COHORTS_ROOT/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet (Y in 2016–2019).")

Step 0: Verify feature importance (required); cohorts and 4_model_data (Step 4 inputs); Step 6 (informational)

  Locations: Cohorts=COHORTS_ROOT, FI=Step 3/3b, ModelData=MODEL_DATA_ROOT, Model=Step 6 outputs

  opioid_ed / 13-24:  Cohorts=True, FI=True, ModelData=True, Model=True  -> ready
  opioid_ed / 25-44:  Cohorts=True, FI=True, ModelData=True, Model=False  -> ready
  opioid_ed / 45-54:  Cohorts=True, FI=True, ModelData=True, Model=False  -> ready
  opioid_ed / 55-64:  Cohorts=True, FI=True, ModelData=True, Model=False  -> ready
  non_opioid_ed / 65-74:  Cohorts=True, FI=True, ModelData=True, Model=False  -> ready
  non_opioid_ed / 75-84:  Cohorts=True, FI=True, ModelData=True, Model=False  -> ready
  non_opioid_ed / 85-94:  Cohorts=True, FI=True, ModelData=True, Model=False  -> ready

All prerequisites are available to build model data. Run Pipeline Step 4–6 cells below.
  (If Step 6 is already built elsewhere, you can sync from S3 or skip those cells.)


## Pipeline Step 4: Model data

Build `model_events.parquet` for each cohort/age_band from Step 2 cohort data and Step 3b feature importance. Outputs go to `MODEL_DATA_ROOT/cohort_name={cohort}/age_band={age_band}/model_events.parquet`. Run the cell below for all PGx cohorts/age_bands defined in this notebook.

In [5]:
# Pipeline Step 4: BUILD model_events.parquet by running create_model_data.py, then QA.
# The script READS: COHORTS_ROOT (cohort.parquet), gold/medical, gold/pharmacy, and feature importance.
# It WRITES: MODEL_DATA_ROOT/cohort_name={cohort}/age_band={age_band}/model_events.parquet
import duckdb

def _model_data_candidates(cohort: str, age_band: str):
    """Canonical location for model_events.parquet (Step 4 writes to MODEL_DATA_ROOT)."""
    return [MODEL_DATA_ROOT]

def _model_data_path(cohort: str, age_band: str) -> Path:
    """Resolve model_events.parquet path (Step 4 writes to get_model_data_root() = DATA_ROOT or PROJECT on Linux)."""
    for base in _model_data_candidates(cohort, age_band):
        p = base / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
        if p.exists():
            return p
    return None

def _log_model_data_qa(cohort: str, age_band: str) -> None:
    """Log location, target distribution, and control:case ratio for model_events.parquet."""
    path = _model_data_path(cohort, age_band)
    if not path:
        print(f"  [WARN] model_events.parquet not found for {cohort}/{age_band}")
        for base in _model_data_candidates(cohort, age_band):
            p = base / f"cohort_name={cohort}" / f"age_band={age_band}" / "model_events.parquet"
            print(f"    Checked: {p}  (exists: {p.exists()})")
        print(f"    Build did not write output. Check script stdout above: [INFO] data roots and example cohort path (exists=?). Layout must be {COHORTS_ROOT}/cohort_name=X/event_year=Y/age_band=Z/cohort.parquet (Y in 2016–2019). Sync cohorts to COHORTS_ROOT if needed, then re-run this cell.")
        return
    print(f"  Location: {path}")
    con = duckdb.connect()
    try:
        dist = con.execute("SELECT target, COUNT(*)::BIGINT AS n FROM read_parquet(?) GROUP BY target ORDER BY target", [str(path)]).fetchall()
        total = sum(row[1] for row in dist)
        by_target = {int(row[0]): int(row[1]) for row in dist}
        n_controls = by_target.get(0, 0)
        n_cases = by_target.get(1, 0)
        ratio = (n_controls / n_cases) if n_cases else 0
        print(f"  Target distribution: {by_target} (total rows: {total:,})")
        print(f"  Control:case ratio: {n_controls:,}:{n_cases:,} = {ratio:.2f}:1")
    finally:
        con.close()

for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 4: {cohort} / {age_band} (building model_events.parquet)")
        r = subprocess.run(
            [sys.executable, "create_model_data.py", "--cohort", cohort, "--age-band", age_band],
            cwd=PROJECT_ROOT / "4_model_data",
            capture_output=False,
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
        _log_model_data_qa(cohort, age_band)
print("Step 4 complete.")

→ Step 4: opioid_ed / 13-24 (building model_events.parquet)
[INFO] Found Step 3b refined feature importance: /home/pgx3874/pgx-analysis/3b_feature_importance_eda/outputs/opioid_ed/13_24/opioid_ed_13_24_cohort_feature_importance.csv
[INFO] Cohort root: tried ['/mnt/nvme/gold/cohorts', '/mnt/nvme/data/gold_cohorts', '/home/pgx3874/pgx-analysis/data/gold_cohorts'] -> using /mnt/nvme/gold/cohorts (first existing)
[INFO] Medical root: tried ['/mnt/nvme/gold/medical', '/mnt/nvme/data/gold_medical', '/home/pgx3874/pgx-analysis/data/gold_medical'] -> using /mnt/nvme/gold/medical (first existing)
[INFO] Pharmacy root: tried ['/mnt/nvme/gold/pharmacy', '/mnt/nvme/data/gold_pharmacy', '/home/pgx3874/pgx-analysis/data/gold_pharmacy'] -> using /mnt/nvme/gold/pharmacy (first existing)
[INFO] Step 4 data roots: cohorts=/mnt/nvme/gold/cohorts, medical=/mnt/nvme/gold/medical, pharmacy=/mnt/nvme/gold/pharmacy
[INFO] Example cohort path (must exist for build to run): /mnt/nvme/gold/cohorts/cohort_name=op

## Pipeline Step 5: PGx analysis

Add PGx features (e.g. CPIC drug counts) to model data. Reads from Step 4 outputs and writes updated model data used by Step 6. Run for each cohort/age_band.

In [6]:
# Pipeline Step 5: run_analysis.py for each REQUIRED_COHORTS (cohort, age_band)
# Set FORCE_STEP5 = True to re-run even when S3 outputs or checkpoints exist
FORCE_STEP5 = True
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 5: {cohort} / {age_band}")
        cmd = [sys.executable, "run_analysis.py", "--cohort-name", cohort, "--age-band", age_band]
        if FORCE_STEP5:
            cmd.append("--force")
        r = subprocess.run(cmd, cwd=PROJECT_ROOT / "5_pgx_analysis")
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 5 complete.")

→ Step 5: opioid_ed / 13-24
2026-02-06 12:51:11,593 - INFO - Runtime environment: os=linux logical_cores=32 ram_gb=1024 fast_root=/mnt/nvme
2026-02-06 12:51:11,593 - INFO - Force re-run: ignoring existing S3 outputs and checkpoints.
2026-02-06 12:51:11,593 - INFO - [FUNCTION][5_pgx_analysis][run_pgx_analysis] START time=2026-02-06 12:51:11 mem_mb=1080.7 cpu_pct=0.0
2026-02-06 12:51:11,593 - INFO - Starting PGx analysis for opioid_ed / 13-24
2026-02-06 12:51:11,593 - INFO - Using global drug-to-CPIC mapping from outputs/global/drug_cpic_mapping_global.csv
2026-02-06 12:51:11,593 - INFO - [STEP][5_pgx_analysis][create_pgx_features] START time=2026-02-06 12:51:11 mem_mb=1080.7 cpu_pct=0.0
2026-02-06 12:51:11,593 - INFO - Creating PGx features for opioid_ed / 13-24
2026-02-06 12:51:13,240 - INFO - PGx features created
2026-02-06 12:51:13,240 - INFO - Create stdout:

Created 2 PGx features for 11776 patients
Output format: Ready for merging with other features (uses mi_person_key)
Saved to:

## Pipeline Step 6: Final model deployment

Train final models per cohort/age_band. Reads Step 4 model data and Step 5 PGx features; writes trained models and `feature_schema.json` to `6_final_model/outputs` (or DATA_ROOT). These outputs are used by "Prepare models" and deployment below.

## Step 1: Train Models — idempotent with checkpoint

Extract valid codes (drugs, ICD, CPT) from feature importance for dashboard dropdowns. Uses Step 3b `cohort_feature_importance` when available, else Step 3 `aggregated_feature_importance`. **Checkpoint:** step is skipped if S3 checkpoint exists.

In [7]:
# Pipeline Step 6: run_final_model.py for each REQUIRED_COHORTS (cohort, age_band)
# Note: script uses --age_band (underscore)
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Step 6: {cohort} / {age_band}")
        r = subprocess.run(
            [sys.executable, "run_final_model.py", "--cohort", cohort, "--age_band", age_band],
            cwd=PROJECT_ROOT / "6_final_model",
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("Step 6 complete.")

→ Step 6: opioid_ed / 13-24
2026-02-06 12:51:54,387 - INFO - Runtime environment: os=linux logical_cores=32 ram_gb=1024 fast_root=/mnt/nvme
2026-02-06 12:51:54,387 - INFO - Step 6 outputs already exist locally for opioid_ed/13-24; skipping regeneration.
2026-02-06 12:51:54,387 - INFO -   Found 8 output files
2026-02-06 12:51:54,435 - INFO - [OK] File already exists in S3: s3://pgxdatalake/gold/final_model/opioid_ed/13-24/opioid_ed_13_24_model_selection_metadata.json (skipping upload)
2026-02-06 12:51:54,444 - INFO - [OK] File already exists in S3: s3://pgxdatalake/gold/final_model/opioid_ed/13-24/opioid_ed_13_24_best_xgboost_model.json (skipping upload)
2026-02-06 12:51:54,453 - INFO - [OK] File already exists in S3: s3://pgxdatalake/gold/final_model/opioid_ed/13-24/opioid_ed_13_24_best_catboost_model.cbm (skipping upload)
2026-02-06 12:51:54,464 - INFO - [OK] File already exists in S3: s3://pgxdatalake/gold/final_model/opioid_ed/13-24/xgboost.joblib (skipping upload)
2026-02-06 12:51:

/home/pgx3874/jupyter-env/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3678: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Step 2: Combine SHAP and FFA results (required)

SHAP (Step 7) and FFA (Step 8) are **mandatory outputs**. Run those steps first, then combine results per cohort/age_band for the Causal Analysis tab.

In [11]:
# Required: combine SHAP (Step 7) and FFA (Step 8) results for Causal Analysis tab.
for cohort, bands in REQUIRED_COHORTS.items():
    for age_band in bands:
        print(f"→ Combine SHAP/FFA: {cohort} / {age_band}")
        r = subprocess.run(
            [sys.executable, "combine_shap_ffa_results.py", "--cohort", cohort, "--age-band", age_band, "--output-dir", str(PROJECT_ROOT / "9_risk_dashboard" / "outputs")],
            cwd=DATA_PREP_DIR,
        )
        if r.returncode != 0:
            raise SystemExit(r.returncode)
print("SHAP/FFA combine complete.")

→ Combine SHAP/FFA: opioid_ed / 13-24


  File "/home/pgx3874/pgx-analysis/9_risk_dashboard/data_preparation/combine_shap_ffa_results.py", line 358
    if args.all_cohorts:
IndentationError: unexpected indent


SystemExit: 1

/home/pgx3874/jupyter-env/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3678: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [9]:
import logging
logger = logging.getLogger(__name__)
if check_step_checkpoint_exists("9_dashboard_metadata", "all", "all", logger):
    print("Step 1 (generate metadata) already completed (checkpoint exists). Skipping.")
else:
    r = subprocess.run([sys.executable, "generate_metadata.py", "--all"], cwd=DATA_PREP_DIR)
    if r.returncode == 0:
        save_step_checkpoint("9_dashboard_metadata", "all", "all", logger=logger)
    if r.returncode != 0:
        raise SystemExit(r.returncode)

Step 1 (generate metadata) already completed (checkpoint exists). Skipping.


## Step 3: Prepare models — idempotent with checkpoint

Package models and feature schemas from `6_final_model/outputs` into `9_risk_dashboard/outputs/models`. **Checkpoint:** step is skipped if S3 checkpoint exists.

In [10]:
import logging
logger = logging.getLogger(__name__)
if check_step_checkpoint_exists("9_dashboard_models", "all", "all", logger):
    print("Step 2 (prepare models) already completed (checkpoint exists). Skipping.")
else:
    r = subprocess.run([sys.executable, "prepare_models.py", "--all"], cwd=DATA_PREP_DIR)
    if r.returncode == 0:
        save_step_checkpoint("9_dashboard_models", "all", "all", logger=logger)
    if r.returncode != 0:
        raise SystemExit(r.returncode)


Preparing models for Lambda Container (ECR) deployment
Models will be placed in: models/
This directory will be copied into Docker container image


Preparing models for opioid_ed

Processing 13-24...
  Extracting feature schema and model weights...
  Using equal weights (1.0 each)
  Saved feature schema (2520 features)
  Model weights included in schema
  Loading catboost model...
  Saved catboost model to: /home/pgx3874/pgx-analysis/9_risk_dashboard/outputs/models/opioid_ed/13_24/catboost.joblib
  Loading xgboost model...
  Saved xgboost model to: /home/pgx3874/pgx-analysis/9_risk_dashboard/outputs/models/opioid_ed/13_24/xgboost.joblib
  Loading xgboost_rf model...
  Complete: /home/pgx3874/pgx-analysis/9_risk_dashboard/outputs/models/opioid_ed/13_24

Processing 25-44...
  Extracting feature schema and model weights...
  Using equal weights (1.0 each)
  Saved feature schema (2342 features)
  Model weights included in schema
  Loading catboost model...
  Saved catboost model to: /hom

## Step 4: Prepare Lambda directory

Assemble `lambda_dir` under `9_risk_dashboard` for Docker build (models, metadata, CPIC data).

In [ ]:
# Check Docker service
import subprocess
import platform

def check_docker():
    """Simple Docker check - verify if Docker is accessible."""
    print("\n" + "=" * 80)
    print("Docker Check")
    print("=" * 80)
    
    try:
        result = subprocess.run(
            ["docker", "ps"],
            capture_output=True,
            text=True,
            timeout=5
        )
        
        if result.returncode == 0:
            print("✓ Docker is running")
            logger.info("Docker is accessible")
            return True
        else:
            print("⚠ Docker is not accessible")
            if "permission denied" in result.stderr.lower():
                print("  Permission issue - you may need to add user to docker group:")
                print("    Linux: sudo usermod -aG docker $USER && newgrp docker")
            else:
                print(f"  Error: {result.stderr.strip()}")
            return False
            
    except FileNotFoundError:
        print("✗ Docker not found - please install Docker")
        system = platform.system()
        if system == "Windows":
            print("  Install Docker Desktop from: https://www.docker.com/products/docker-desktop")
        else:
            print("  Linux: https://docs.docker.com/engine/install/")
            print("  macOS: Install Docker Desktop")
        return False
    except Exception as e:
        print(f"⚠ Error checking Docker: {e}")
        return False
    
    print("=" * 80)

# Run check
docker_ok = check_docker()

In [ ]:
# Run from DEPLOY_DIR so prepare_lambda_dir.py finds paths correctly (no shell var expansion in notebook)
r = subprocess.run([sys.executable, "prepare_lambda_dir.py"], cwd=DEPLOY_DIR)
if r.returncode != 0:
    raise SystemExit(r.returncode)
print("Lambda directory prepared.")

## Step 5: Verify Lambda directory

Ensure all required files are present before building the image.

In [ ]:
subprocess.run([sys.executable, "prepare_lambda_dir.py", "--verify-only"], cwd=DEPLOY_DIR, check=True)

## Step 6: Build and deploy

Build the Docker image and push to ECR; then update API Gateway/Lambda. Use the deployment script in `9_risk_dashboard/deployment`.

In [ ]:
# From 9_risk_dashboard directory:
# ./deployment/docker_build.sh
# Or manually:
# docker build -t pgx-risk-dashboard .
# Then push to ECR and update Lambda function (see docs/Step10_Results/README_results_deployment.md)
print("Run from shell: cd 9_risk_dashboard && ./deployment/docker_build.sh")
print("See: docs/Step10_Results/README_results_deployment.md")